# MeChess 4/4: label annotated moves with the trained language model (GPU notebook)

Only an adapter: it runs `chessme books-preflight` and `chessme books-nlp-label`. The same work on a laptop:
`python -m chessme books-nlp-label --data data/books_learn --model-dir data/books_learn/run/nlp` (slower, no GPU needed).

Every annotated move that has a readable comment gets **concepts** (from the keyword lexicon and from the model reading the comment with the keywords hidden), the **verdict** the comment implies
and **who stands better**, attached to the position it was written about. The output, `labelled.jsonl.gz`, is the training data for a model that looks at positions (concept per position).
A **report** lists, per concept, how often the lexicon and the model find it and how well the model finds what the lexicon finds; it also lists a sample of concepts only the model found, to check by eye.

**Before you run**
1. Add two inputs (*Add Input -> Notebook output files*): the output of notebook 1 (the collected data) and the output of notebook 2 (the trained model). Leave `DATA_DIR` and `MODEL_DIR` empty to find them automatically, or set them to the folders
   (for example `/kaggle/input/notebooks/<your-username>/<notebook-name>`); the notebook prints what it found and stops at once if either is missing.
2. *Accelerator -> GPU*, *Internet -> On* (the base encoder is downloaded), set `REPO_URL`, **Save Version -> Save & Run All (Commit)**.
3. Step 1 checks the GPU and the dependencies; step 2 labels the first 2,000 moves and prints the report (about a minute) so a broken setup costs minutes, not the quota.
4. Step 3 is the real run. It saves its progress as it goes and stops cleanly at `BUDGET_MIN`. If it says it paused, run the notebook again with **this notebook's earlier output added as an input**: it continues where it stopped.

In [ ]:
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here
BASE_MODEL = "distilroberta-base"                       # the encoder the model was trained on (see the training notebook)
DATA_DIR = ""                                           # the data notebook's output folder; empty: search /kaggle/input
MODEL_DIR = ""                                          # the training notebook's output folder (the one holding nlp/model.pt); empty: search /kaggle/input
VARIANT = "composite"                                   # "composite": the model kept on all three heads; "concepts": the one best on concepts alone (if the run has it)
BATCH = 128
BUDGET_MIN = 600                                        # wall-clock budget of the real run, minutes
OUT = "/kaggle/working/labels" if os.path.exists("/kaggle") else "labels_local"

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):                      # Kaggle: fetch the repository (Internet must be on)
    if "<you>" in REPO_URL:
        raise SystemExit("Set REPO_URL to your repository first.")
    sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

INPUTS = ["--input-root", "/kaggle/input"] + (["--data-dir", DATA_DIR] if DATA_DIR else []) + (["--model-dir", MODEL_DIR] if MODEL_DIR else [])
LABEL_OUT = f"{OUT}/labelled/labelled.jsonl.gz"

def sh_paused(*args):
    """Like sh, but exit code 3 (the time budget was reached) is not an error: it is reported and the notebook ends normally."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc not in (0, 3):
        raise RuntimeError(f"failed: {' '.join(args)}")
    return rc

## Step 1. Preflight: a GPU is present, dependencies import, the base encoder can be downloaded

In [ ]:
sh(*CLI, "books-preflight", "--out", OUT, "--need-gpu", "--backend", "transformer", "--model", BASE_MODEL, "--no-network")

## Step 2. Trial on 2,000 moves: finds the data and the model, labels, prints the report (about a minute)

In [ ]:
sh(*CLI, "books-nlp-label", *INPUTS, "--variant", VARIANT, "--limit", "2000", "--sample", "10", "--out", f"{OUT}/trial/labelled.jsonl.gz", "--log", f"{OUT}/trial.log")
print(pathlib.Path(OUT, "trial", "label_report.md").read_text())

## Step 3. The real run (saves progress; stops cleanly at the time budget)

In [ ]:
rc = sh_paused(*CLI, "books-nlp-label", *INPUTS, "--variant", VARIANT, "--batch", str(BATCH), "--deadline-minutes", str(BUDGET_MIN), "--sample", "60", "--out", LABEL_OUT, "--log", f"{OUT}/labelled/label.log")
print("PAUSED at the time budget: run this notebook again with its own earlier output as an input." if rc == 3 else "finished")

## Step 4. Report (per concept: lexicon vs model, and a sample to check by eye)

In [ ]:
r = pathlib.Path(OUT, "labelled", "label_report.md")
print(r.read_text() if r.exists() else "no report yet: the run is paused")